## 1. Importation et Nettoyage des Données Réelles


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Chargement du jeu de données officiel des accidents d'aviation
url = 'https://raw.githubusercontent.com/jldbc/datasets/master/AviationData.csv'
try:
    df_raw = pd.read_csv(url, encoding='ISO-8859-1', low_memory=False)
    print("Jeu de données réel chargé avec succès !")
except Exception as e:
    print("Erreur lors du chargement, utilisation d'une structure alternative :", e)

# Sélection et renommage des colonnes stratégiques pour correspondre à notre étude
df = df_raw[['Event.Date', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Make', 'Model']].copy()
df.columns = ['Date', 'Fatalities', 'Serious_Injuries', 'Make', 'Model']

# Nettoyage
df['Fatalities'] = pd.to_numeric(df['Fatalities'], errors='coerce').fillna(0)
df['Serious_Injuries'] = pd.to_numeric(df['Serious_Injuries'], errors='coerce').fillna(0)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])

# Extraction temporelle
df['Year'] = df['Date'].dt.year
df['Decade'] = (df['Year'] // 10) * 10

# Filtrer sur une période pertinente pour avoir de la consistance (jusqu'à présent)
df = df[(df['Year'] >= 1950) & (df['Year'] <= 2023)]

print("Aperçu des données réelles nettoyées :")
print(df.head())

## 2. Analyse Exploratoire des Données Réelles (EDA)


In [ ]:
total_crashes = len(df)
total_fatalities = df['Fatalities'].sum()
total_serious = df['Serious_Injuries'].sum()

print(f"Nombre total d'accidents réels analysés : {total_crashes}")
print(f"Total cumulé des décès : {int(total_fatalities):,}")
print(f"Total cumulé des blessés graves : {int(total_serious):,}")

## 3. Analyses Statistiques Inférentielles (SciPy)


In [ ]:
fatalities_1980 = df[df['Decade'] == 1980]['Fatalities']
fatalities_2010 = df[df['Decade'] == 2010]['Fatalities']

print(f"Moyenne de décès/accident (1980s) : {fatalities_1980.mean():.2f}")
print(f"Moyenne de décès/accident (2010s) : {fatalities_2010.mean():.2f}\n")

stat_t, p_val = stats.ttest_ind(fatalities_1980, fatalities_2010, equal_var=False)
print(f"Statistique T du test : {stat_t:.4f}")
print(f"P-valeur du test : {p_val:.4e}")

if p_val < 0.05:
    print("Conclusion : Rejet de l'hypothèse nulle. La différence de mortalité par accident entre ces deux décennies est statistiquement significative.")
else:
    print("Conclusion : Échec du rejet de l'hypothèse nulle. Pas de différence statistiquement significative.")

## 4. Visualisations Graphiques Explicatives


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Évolution temporelle réelle
df_yearly = df.groupby('Year').size().reset_index(name='Count')
sns.lineplot(data=df_yearly, x='Year', y='Count', ax=axes[0], color='darkred', linewidth=2.5)
axes[0].set_title("Évolution historique du nombre annuel d'accidents d'avion (Données Réelles)", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Nombre d'accidents")

# Évolution des décès par décennie
df_decade_profit = df.groupby('Decade')['Fatalities'].mean().reset_index()
sns.barplot(data=df_decade_profit, x='Decade', y='Fatalities', ax=axes[1], palette='Blues_d')
axes[1].set_title("Nombre moyen de décès par accident selon la décennie", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Moyenne des décès")

plt.tight_layout()
plt.show()